<div align="center">

# SPEKTRAN — Quick Start
### The MNIST of Gas Sensing

**Open-source simulation engine + ML benchmark for optical spectroscopy**

[![GitHub](https://img.shields.io/github/stars/spektran/spektran?style=social)](https://github.com/spektran/spektran)
[![PyPI](https://img.shields.io/pypi/v/spektran?style=flat-square&color=blue)](https://pypi.org/project/spektran/)
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.21790394-blue?style=flat-square)](https://doi.org/10.5281/zenodo.21790394)

</div>

---

This notebook walks you through SPEKTRAN in **5 minutes**:

1. **Simulate** a CH4 absorption spectrum from HITRAN physics
2. **Load** official benchmark splits from Hugging Face
3. **Train** a Ridge regression baseline (T1 — concentration prediction)
4. **Evaluate** cross-instrument generalization (T3 — held-out instruments)
5. **Discover** the overfitting-vs-complexity finding

> **You don't need to be a spectroscopist.** If you work on regression, domain generalization, or scientific ML, this benchmark is for you.

## 0. Install

In [ ]:
!pip install -q spektran scikit-learn matplotlib

## 1. Simulate a CH4 Absorption Spectrum

SPEKTRAN wraps HITRAN line-by-line physics into a simple API. Let's simulate methane (CH4) absorption in the 2ν3 band — the same spectral region used by real TDLAS sensors for natural gas leak detection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from spektran.physics import simulate_absorbance

# Simulate CH4 at three concentrations
concentrations = [50, 100, 200]  # ppm
colors = ['#2196F3', '#FF9800', '#E91E63']

fig, ax = plt.subplots(figsize=(10, 4))
for conc, color in zip(concentrations, colors):
    nu, absorbance = simulate_absorbance(
        molecule="CH4",
        concentration_ppm=conc,
        temperature_K=296.0,
        pressure_atm=1.0,
        path_length_m=10.0,
        wavenumber_start_cm1=6046.0,
        wavenumber_end_cm1=6048.0,
    )
    ax.plot(nu, absorbance, color=color, label=f'{conc} ppm')

ax.set_xlabel('Wavenumber (cm⁻¹)')
ax.set_ylabel('Absorbance')
ax.set_title('CH4 Absorption — 2ν₃ Band (HITRAN Physics)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Spectral points: {len(nu)}")
print(f"Peak absorbance at 100 ppm: {absorbance.max():.4e}")

## 2. Load Benchmark Data

SPEKTRAN provides 9 benchmark tasks with official train/val/test splits. The data lives on Hugging Face — no local generation needed.

We'll work with **T1 (Concentration Regression)**: given a noisy raw scan from a virtual TDLAS instrument, predict CH4 concentration in ppm.

In [ ]:
from datasets import load_dataset

ds = load_dataset("spektran/spektran-ch4-v0")
print(f"Splits: {list(ds.keys())}")
print(f"Train: {len(ds['t1-train'])} records")
print(f"Val:   {len(ds['t1-val'])} records")
print(f"Test:  {len(ds['t1-test'])} records")
print(f"\nHeld-out instruments (T3): {len(ds['t3-test-heldout'])} records")

# Peek at a single record
sample = ds['t1-train'][0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"Scan length: {len(sample['raw_scan'])} points")
print(f"True concentration: {sample['concentration_ppm']:.1f} ppm")

In [ ]:
# Prepare numpy arrays
def split_to_arrays(split):
    X = np.array(split['raw_scan'])
    y = np.array(split['concentration_ppm'])
    return X, y

X_train, y_train = split_to_arrays(ds['t1-train'])
X_val, y_val = split_to_arrays(ds['t1-val'])
X_test, y_test = split_to_arrays(ds['t1-test'])
X_heldout, y_heldout = split_to_arrays(ds['t3-test-heldout'])

print(f"X_train shape: {X_train.shape}  (records × spectral points)")
print(f"y_train range: [{y_train.min():.0f}, {y_train.max():.0f}] ppm")

In [ ]:
# Visualize a few training scans
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for i, ax in enumerate(axes):
    idx = i * len(X_train) // 3
    ax.plot(X_train[idx], color='#333', linewidth=0.8)
    ax.set_title(f'{y_train[idx]:.0f} ppm')
    ax.set_xlabel('Point index')
    if i == 0:
        ax.set_ylabel('Raw scan signal')
    ax.grid(True, alpha=0.3)

fig.suptitle('Sample Raw Scans (T1 — noisy virtual instrument output)', y=1.02)
plt.tight_layout()
plt.show()

## 3. Train a Ridge Baseline (T1)

The simplest baseline: standardize features → Ridge regression with cross-validated α.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)

# Select alpha on validation split
best_alpha, best_mae = None, np.inf
for alpha in [0.1, 1.0, 10.0, 100.0, 1000.0]:
    model = Ridge(alpha=alpha).fit(X_train_s, y_train)
    preds = model.predict(X_val_s)
    mae = np.mean(np.abs(preds - y_val))
    print(f"  α={alpha:>6g}  →  val MAE = {mae:.3f} ppm")
    if mae < best_mae:
        best_alpha, best_mae = alpha, mae

print(f"\n✓ Selected α = {best_alpha}, val MAE = {best_mae:.3f} ppm")

In [ ]:
# Retrain with best alpha and evaluate on test
model = Ridge(alpha=best_alpha).fit(X_train_s, y_train)

# T1: same-distribution test
y_pred_t1 = model.predict(scaler.transform(X_test))
mae_t1 = np.mean(np.abs(y_pred_t1 - y_test))

# T3: held-out instruments (unseen during training)
y_pred_t3 = model.predict(scaler.transform(X_heldout))
mae_t3 = np.mean(np.abs(y_pred_t3 - y_heldout))

degradation = mae_t3 / mae_t1

print(f"T1 test MAE:  {mae_t1:.2f} ppm  (same instruments)")
print(f"T3 test MAE:  {mae_t3:.2f} ppm  (held-out instruments)")
print(f"Degradation:  {degradation:.2f}x")

## 4. The Overfitting-vs-Complexity Finding

Here's SPEKTRAN's headline result — and the challenge for the ML community:

| Model | T1 MAE ↓ | T3 MAE ↓ | T3 Degradation |
|:------|:--------:|:--------:|:--------------:|
| Ridge regression | **2.84** | **3.72** | **1.31x** |
| Patchified Transformer | 7.39 | 10.81 | 1.46x |
| 1D CNN | 15.58 | 28.30 | 1.82x |

> **Model complexity correlates with instrument overfitting.**
> Ridge (linear) degrades only 1.31x on unseen instruments.
> The Transformer degrades 1.46x. The CNN degrades 1.82x.
>
> Deep models learn instrument-specific noise patterns
> instead of the underlying physics. **Can you build a model that breaks this pattern?**

In [ ]:
# Visualize the prediction quality
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_true, y_pred, title in [
    (axes[0], y_test, y_pred_t1, f'T1 — Same Instruments (MAE={mae_t1:.2f})'),
    (axes[1], y_heldout, y_pred_t3, f'T3 — Held-Out Instruments (MAE={mae_t3:.2f})'),
]:
    ax.scatter(y_true, y_pred, alpha=0.3, s=10, color='#1565C0')
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, '--', color='#E53935', linewidth=1.5, label='Perfect')
    ax.set_xlabel('True concentration (ppm)')
    ax.set_ylabel('Predicted concentration (ppm)')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 5. Your Turn — Beat the Baselines!

The data is ready. The evaluation is standardized. **Can you do better?**

Ideas to explore:
- **Domain-invariant representations** — adversarial training, DANN, or instrument-blind features
- **Physics-informed priors** — embed Beer-Lambert law as an inductive bias
- **Spectral preprocessing** — wavelet denoising, derivative spectra, peak-area integration
- **Multi-task learning** — jointly predict concentration + denoise (T1+T2)

### Submit to the Leaderboard

```python
# Evaluate with the official CLI
# !pip install spektran
# !spektran benchmark --task T1-concentration --truth test.h5 --predictions your_preds.csv
```

Submit results → [**Leaderboard**](https://spektran.github.io/spektran/leaderboard/#submitting-results)

---

### Links

- [GitHub](https://github.com/spektran/spektran) — Star us if this was useful! ⭐
- [Full documentation](https://spektran.github.io/spektran/)
- [Dataset on Hugging Face](https://huggingface.co/datasets/spektran/spektran-ch4-v0)
- [Paper / DOI](https://doi.org/10.5281/zenodo.21790394)
- [Interactive Demo](https://huggingface.co/spaces/spektran/spektran-demo)